In [1]:
import pandas as pd

# 1. Cargar el dataset meteorológico crudo
# (Asegúrate de poner el nombre real de tu archivo)
df_clima = pd.read_csv('../data/2024hourlymeteo.csv')

# 2. Crear la columna 'Timestamp' fusionando Año, Mes, Día y Hora
# Pandas es tan listo que si le pasas estas 4 columnas, genera la fecha automáticamente
df_clima['Timestamp'] = pd.to_datetime(df_clima[['year', 'month', 'day', 'hour']])

# 3. Renombrar las columnas para que queden en español y claras
df_clima = df_clima.rename(columns={
    'temp': 'Temperatura',
    'prcp': 'Lluvia'
})

# 4. Aislar SOLO las columnas que importan para tu Tabla Maestra
df_clima_limpio = df_clima[['Timestamp', 'Temperatura', 'Lluvia']]

# Mostrar el resultado para comprobar
print("¡Limpieza meteorológica completada!")
df_clima_limpio.head()

c:\Users\jordi\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


¡Limpieza meteorológica completada!


,Timestamp,Temperatura,Lluvia
0,2024-01-01 00:00:00,7.5,0.0
1,2024-01-01 01:00:00,8.0,0.0
2,2024-01-01 02:00:00,7.7,0.0
3,2024-01-01 03:00:00,7.5,0.0
4,2024-01-01 04:00:00,7.7,0.0


In [2]:
df_clima_limpio.to_csv('../data/clima_limpio.csv', index=False)


In [3]:
import pandas as pd

# 1. Cargamos el dataset crudo
df_energia = pd.read_csv('../data/consum_electric/hora_poblenou.csv', sep=';')

columnas_utiles = ['Fecha y hora de la medida', 'Medida de la magnitud Activa Entrante (neta)']
df_energia_limpio = df_energia[columnas_utiles].copy()

df_energia_limpio.rename(columns={
    'Fecha y hora de la medida': 'Timestamp',
    'Medida de la magnitud Activa Entrante (neta)': 'Consumo_kWh'
}, inplace=True)

df_energia_limpio['Timestamp'] = df_energia_limpio['Timestamp'].str.strip()

# --- EL FIX DEFINITIVO ---

# 1. ELIMINAMOS la hora 25 (el ruido del cambio de hora)
df_energia_limpio = df_energia_limpio[~df_energia_limpio['Timestamp'].str.endswith('25:00:00')].copy()

# 2. Identificamos las filas que tienen el '24:00:00' (final del día)
mask_24h = df_energia_limpio['Timestamp'].str.endswith('24:00:00')

# 3. Cambiamos temporalmente el '24:00' por '00:00' para que Pandas lo lea
df_energia_limpio['Timestamp'] = df_energia_limpio['Timestamp'].str.replace('24:00:00', '00:00:00')

# 4. Convertimos a Fecha/Hora real
df_energia_limpio['Timestamp'] = pd.to_datetime(df_energia_limpio['Timestamp'], format='%d/%m/%Y %H:%M:%S')

# 5. A las que marcaban '24:00', les sumamos 1 día para moverlas a la medianoche correcta
df_energia_limpio.loc[mask_24h, 'Timestamp'] += pd.Timedelta(days=1)

print("¡Limpieza de Energía completada sin horas duplicadas!")
df_energia_limpio.head()

¡Limpieza de Energía completada sin horas duplicadas!


,Timestamp,Consumo_kWh
0,2024-01-01 01:00:00,96
1,2024-01-01 02:00:00,96
2,2024-01-01 03:00:00,97
3,2024-01-01 04:00:00,96
4,2024-01-01 05:00:00,95


In [4]:
import pandas as pd
import numpy as np
total_aules = 94

# 1. Carregar les dades
# Nota: Assegura't de tenir instal·lada la llibreria 'openpyxl' (pip install openpyxl)
df = pd.read_excel('../data/aules/2024Tanger.xlsx')

# 2. Crear columnes de Timestamp reals combinant data i hora
df['Inici'] = pd.to_datetime(df['Data inicial'] + ' ' + df['Hora inicial'], dayfirst=True)
df['Final'] = pd.to_datetime(df['Data final'] + ' ' + df['Hora final'], dayfirst=True)

# 3. Definir el rang horari complet del fitxer (des de la primera fins a l'última reserva)
inici_simulacio = df['Inici'].min().replace(minute=0, second=0)
final_simulacio = df['Final'].max().replace(minute=0, second=0) + pd.Timedelta(hours=1)

hores = pd.date_range(start=inici_simulacio, end=final_simulacio, freq='H')

resultats = []

# 4. Iterar per cada hora per calcular l'ocupació
for hora in hores:
    hora_fi = hora + pd.Timedelta(hours=1)
    
    # Una aula està ocupada si la reserva comença abans que acabi l'hora 
    # I acaba després que comenci l'hora
    ocupades = df[(df['Inici'] < hora_fi) & (df['Final'] > hora)]
    
    num_aules_ocupades = ocupades['Espai'].nunique()
    percentatge = (num_aules_ocupades / total_aules) * 100

    resultats.append({
        'Timestamp': hora,
        'Aules_Ocupades': num_aules_ocupades,
        'Ocupacio_Percent': round(percentatge, 2)
    })

# 5. Crear el DataFrame final
df_ocupacio = pd.DataFrame(resultats)



# --- ÚS DEL CODI ---
# ruta = "les_teves_reserves.xlsx"

# Guardar el resultat per fer-lo servir en la Fase 2 del TFG
df_ocupacio.to_csv("../data/ocupacio_aules_horaria.csv", index=False)
print(df_ocupacio.head())

C:\Users\jordi\AppData\Local\Temp\ipykernel_1220\2316411811.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hores = pd.date_range(start=inici_simulacio, end=final_simulacio, freq='H')


            Timestamp  Aules_Ocupades  Ocupacio_Percent
0 2024-01-04 10:00:00               1              1.06
1 2024-01-04 11:00:00               2              2.13
2 2024-01-04 12:00:00               1              1.06
3 2024-01-04 13:00:00               1              1.06
4 2024-01-04 14:00:00               1              1.06


Merge

In [5]:
df_calendari = pd.read_csv('../data/Calendari-UPF-2024.csv')
aforament_df = pd.read_csv('../data/ocupacio_aules_horaria.csv')
df_energia_limpio.head(5)

,Timestamp,Consumo_kWh
0,2024-01-01 01:00:00,96
1,2024-01-01 02:00:00,96
2,2024-01-01 03:00:00,97
3,2024-01-01 04:00:00,96
4,2024-01-01 05:00:00,95


In [6]:
df_ocupacio.head(5)


,Timestamp,Aules_Ocupades,Ocupacio_Percent
0,2024-01-04 10:00:00,1,1.06
1,2024-01-04 11:00:00,2,2.13
2,2024-01-04 12:00:00,1,1.06
3,2024-01-04 13:00:00,1,1.06
4,2024-01-04 14:00:00,1,1.06


In [7]:
df_calendari.head(5)


,fecha,tipus_dia
0,2024-01-01,Vacances
1,2024-01-02,Vacances
2,2024-01-03,Vacances
3,2024-01-04,Vacances
4,2024-01-05,Vacances


In [8]:
aforament_df.head(5)


,Timestamp,Aules_Ocupades,Ocupacio_Percent
0,2024-01-04 10:00:00,1,1.06
1,2024-01-04 11:00:00,2,2.13
2,2024-01-04 12:00:00,1,1.06
3,2024-01-04 13:00:00,1,1.06
4,2024-01-04 14:00:00,1,1.06


In [9]:
df_clima_limpio.head(5)


,Timestamp,Temperatura,Lluvia
0,2024-01-01 00:00:00,7.5,0.0
1,2024-01-01 01:00:00,8.0,0.0
2,2024-01-01 02:00:00,7.7,0.0
3,2024-01-01 03:00:00,7.5,0.0
4,2024-01-01 04:00:00,7.7,0.0


In [12]:
# 1. Aseguramos formato datetime en todas (por si acaso)
df_clima_limpio['Timestamp'] = pd.to_datetime(df_clima_limpio['Timestamp'])
df_ocupacio['Timestamp'] = pd.to_datetime(df_ocupacio['Timestamp'])
df_energia_limpio['Timestamp'] = pd.to_datetime(df_energia_limpio['Timestamp'])

# (Si tienes tu tabla de personas Wi-Fi real, asegúrate de que también sea datetime)
# df_wifi['Timestamp'] = pd.to_datetime(df_wifi['Timestamp'])

# 2. EL MERGE MAESTRO: Usamos Energía como base (LEFT JOIN)
df_final = (df_energia_limpio
            .merge(df_clima_limpio, on='Timestamp', how='left')
            .merge(df_ocupacio, on='Timestamp', how='left'))

# NOTA: He quitado 'aforament_df' porque estaba duplicando las aulas. 
# Si tenías una tabla separada para el Wi-Fi, añádela aquí con otra línea:


# 3. RELLENAR LOS VACÍOS CON CEROS
# Como a las 02:00 AM no hay reservas, Pandas pone 'NaN'. Lo convertimos a 0.
df_final['Aules_Ocupades'] = df_final['Aules_Ocupades'].fillna(0)
df_final['Ocupacio_Percent'] = df_final['Ocupacio_Percent'].fillna(0)

# (Opcional) Ordenar las columnas para que sea fácil de leer
columnas_orden = ['Timestamp', 'Temperatura', 'Lluvia', 'Aules_Ocupades', 'Ocupacio_Percent', 'Consumo_kWh']
# Solo reordenamos si esas columnas existen en el df_final
df_final = df_final[[col for col in columnas_orden if col in df_final.columns]]

df_final.head(60)

C:\Users\jordi\AppData\Local\Temp\ipykernel_1220\1592442509.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clima_limpio['Timestamp'] = pd.to_datetime(df_clima_limpio['Timestamp'])


,Timestamp,Temperatura,Lluvia,Aules_Ocupades,Ocupacio_Percent,Consumo_kWh
0,2024-01-01 01:00:00,8.0,0.0,0.0,0.0,96
1,2024-01-01 02:00:00,7.7,0.0,0.0,0.0,96
2,2024-01-01 03:00:00,7.5,0.0,0.0,0.0,97
3,2024-01-01 04:00:00,7.7,0.0,0.0,0.0,96
4,2024-01-01 05:00:00,7.6,0.0,0.0,0.0,95
5,2024-01-01 06:00:00,7.2,0.0,0.0,0.0,106
6,2024-01-01 07:00:00,7.5,0.0,0.0,0.0,97
7,2024-01-01 08:00:00,7.7,0.0,0.0,0.0,98
8,2024-01-01 09:00:00,8.2,0.0,0.0,0.0,103
9,2024-01-01 10:00:00,8.7,0.0,0.0,0.0,100


In [13]:
import pandas as pd

# 1. Assegurem que les dates siguin en el format correcte
# 'df_calendari' és el teu DataFrame amb columnes com 'Data' i 'Tipus_Dia' (Lectiu, Festiu, etc.)
df_calendari['Data'] = pd.to_datetime(df_calendari['fecha'])
df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'])

# 2. CREEM EL PONT: Extraiem només el dia de la taula horària
# Això converteix "2024-01-01 14:00:00" en "2024-01-01"
df_final['Data_Pont'] = df_final['Timestamp'].dt.normalize()

# 3. FEM EL MERGE (LEFT JOIN)
# Unim les dades del calendari a cada hora que correspongui a aquell dia
df_final = pd.merge(
    df_final, 
    df_calendari, 
    left_on='Data_Pont', 
    right_on='Data', 
    how='left'
)

# 4. NETEJA
# Eliminem les columnes auxiliars que ja no necessitem
df_final.drop(columns=['Data_Pont', 'Data'], inplace=True)

# Opcional: Si el calendari té buits, omplim amb "Desconegut" o el valor més comú
df_final['tipus_dia'] = df_final['tipus_dia'].fillna('Lectiu') 

print("✅ Calendari integrat correctament!")
df_final.head(24) # Ara veuràs una columna nova amb el tipus de dia repetit per a cada hora

✅ Calendari integrat correctament!


,Timestamp,Temperatura,Lluvia,Aules_Ocupades,Ocupacio_Percent,Consumo_kWh,fecha,tipus_dia
0,2024-01-01 01:00:00,8.0,0.0,0.0,0.0,96,2024-01-01,Vacances
1,2024-01-01 02:00:00,7.7,0.0,0.0,0.0,96,2024-01-01,Vacances
2,2024-01-01 03:00:00,7.5,0.0,0.0,0.0,97,2024-01-01,Vacances
3,2024-01-01 04:00:00,7.7,0.0,0.0,0.0,96,2024-01-01,Vacances
4,2024-01-01 05:00:00,7.6,0.0,0.0,0.0,95,2024-01-01,Vacances
5,2024-01-01 06:00:00,7.2,0.0,0.0,0.0,106,2024-01-01,Vacances
6,2024-01-01 07:00:00,7.5,0.0,0.0,0.0,97,2024-01-01,Vacances
7,2024-01-01 08:00:00,7.7,0.0,0.0,0.0,98,2024-01-01,Vacances
8,2024-01-01 09:00:00,8.2,0.0,0.0,0.0,103,2024-01-01,Vacances
9,2024-01-01 10:00:00,8.7,0.0,0.0,0.0,100,2024-01-01,Vacances


In [17]:
import pandas as pd
df_simulacion = pd.read_csv('../data/ocupacion_por_hora.csv') # Assegura't de posar el nom correcte del teu fitxer
df_simulacion['Timestamp'] = pd.to_datetime(df_simulacion['Timestamp'])

# 1. AÑADIR EL DÍA DE LA SEMANA
# Creamos una columna numérica (0=Lunes, 6=Domingo) para el Machine Learning
df_final['Dia_Semana_Num'] = df_final['Timestamp'].dt.dayofweek

# Creamos una columna de texto solo para que a ti te sea más fácil leer la tabla
dias_map = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df_final['Dia_Semana'] = df_final['Dia_Semana_Num'].map(dias_map)

# 2. AÑADIR LA OCUPACIÓN SIMULADA (WI-FI)
# Hacemos un LEFT JOIN para pegar la columna 'Ocupacion_Simulada' a nuestra tabla
df_final = pd.merge(
    df_final, 
    df_simulacion[['Timestamp', 'Personas_Reales']], # Solo traemos el tiempo y la ocupación
    on='Timestamp', 
    how='left'
)

df_final.rename(columns={'Personas_Reales': 'Ocupacion_Simulada'}, inplace=True)
df_final['Ocupacion_Simulada'] = df_final['Ocupacion_Simulada'].fillna(0)

# 3. LIMPIEZA FINAL DE COLUMNAS (Eliminamos el "Ruido")
# Borramos 'Aules_Ocupades' para evitar multicolinealidad y 'fecha' que ya no sirve
df_final.drop(columns=['Aules_Ocupades', 'fecha'], inplace=True, errors='ignore')

# 4. REORDENAR PARA QUE QUEDE PERFECTO
columnas_ordenadas = [
    'Timestamp', 
    'Dia_Semana', 
    'tipus_dia', 
    'Temperatura', 
    'Lluvia', 
    'Ocupacio_Percent', 
    'Ocupacion_Simulada', 
    'Consumo_kWh'
]
df_final = df_final[columnas_ordenadas]

print("¡Dataset final optimizado para Machine Learning!")
df_final.head(10)

¡Dataset final optimizado para Machine Learning!


,Timestamp,Dia_Semana,tipus_dia,Temperatura,Lluvia,Ocupacio_Percent,Ocupacion_Simulada,Ocupacion_Simulada,Consumo_kWh
0,2024-01-01 01:00:00,Lunes,Vacances,8.0,0.0,0.0,0.0,0.0,96
1,2024-01-01 02:00:00,Lunes,Vacances,7.7,0.0,0.0,0.0,0.0,96
2,2024-01-01 03:00:00,Lunes,Vacances,7.5,0.0,0.0,0.0,0.0,97
3,2024-01-01 04:00:00,Lunes,Vacances,7.7,0.0,0.0,0.0,0.0,96
4,2024-01-01 05:00:00,Lunes,Vacances,7.6,0.0,0.0,0.0,0.0,95
5,2024-01-01 06:00:00,Lunes,Vacances,7.2,0.0,0.0,0.0,0.0,106
6,2024-01-01 07:00:00,Lunes,Vacances,7.5,0.0,0.0,7.0,7.0,97
7,2024-01-01 08:00:00,Lunes,Vacances,7.7,0.0,0.0,91.0,91.0,98
8,2024-01-01 09:00:00,Lunes,Vacances,8.2,0.0,0.0,160.0,160.0,103
9,2024-01-01 10:00:00,Lunes,Vacances,8.7,0.0,0.0,243.0,243.0,100


In [22]:
# 1. EL RECUENTO GENERAL: ¿En qué columnas faltan datos?
print("--- RECUENTO DE NULOS POR COLUMNA ---")
nulos_por_columna = df_final.isnull().sum()
print(nulos_por_columna[nulos_por_columna > 0]) # Solo muestra las que tienen errores

# 2. LA INSPECCIÓN VISUAL: Ver las filas exactas que están rotas
filas_con_nulos = df_final[df_final.isnull().any(axis=1)]

print(f"\n--- FILAS AFECTADAS ---")
print(f"Hay un total de {len(filas_con_nulos)} filas con al menos un dato faltante.")
print("Aquí tienes una muestra de esas horas conflictivas:")

# Mostramos las primeras 10 filas que tienen algún NaN
display(filas_con_nulos)

--- RECUENTO DE NULOS POR COLUMNA ---
Temperatura      1
Lluvia         252
dtype: int64

--- FILAS AFECTADAS ---
Hay un total de 252 filas con al menos un dato faltante.
Aquí tienes una muestra de esas horas conflictivas:


,Timestamp,Dia_Semana,tipus_dia,Temperatura,Lluvia,Ocupacio_Percent,Ocupacion_Simulada,Ocupacion_Simulada,Consumo_kWh
5759,2024-08-28 01:00:00,Miércoles,Altre,25.0,NaN,0.0,0.0,0.0,96
5760,2024-08-28 02:00:00,Miércoles,Altre,24.7,NaN,0.0,0.0,0.0,98
5761,2024-08-28 03:00:00,Miércoles,Altre,24.5,NaN,0.0,0.0,0.0,96
5762,2024-08-28 04:00:00,Miércoles,Altre,23.7,NaN,0.0,0.0,0.0,96
5763,2024-08-28 05:00:00,Miércoles,Altre,23.6,NaN,0.0,0.0,0.0,96
...,...,...,...,...,...,...,...,...,...
8337,2024-12-13 11:00:00,Viernes,Avaluacio,13.0,NaN,0.0,1040.0,1040.0,350
8338,2024-12-13 12:00:00,Viernes,Avaluacio,14.2,NaN,0.0,1127.0,1127.0,378
8339,2024-12-13 13:00:00,Viernes,Avaluacio,15.6,NaN,0.0,1002.0,1002.0,389
8340,2024-12-13 14:00:00,Viernes,Avaluacio,14.7,NaN,0.0,835.0,835.0,422


In [23]:
# 1. ¿A qué hora exacta falló la Temperatura?
print("🌡️ --- FALLO DE TEMPERATURA ---")
horas_temp = df_final[df_final['Temperatura'].isnull()]['Timestamp']
for h in horas_temp:
    print(f"Falta el dato del: {h}")

# 2. ¿Qué días falló el sensor de Lluvia?
print("\n🌧️ --- FALLOS DEL SENSOR DE LLUVIA ---")
filas_lluvia = df_final[df_final['Lluvia'].isnull()]

# Extraemos la fecha (sin la hora) y contamos cuántas horas falló ese día
dias_fallo_lluvia = filas_lluvia['Timestamp'].dt.date.value_counts().sort_index()

print(f"El sensor de lluvia falló en {len(dias_fallo_lluvia)} días distintos. Resumen:")
for dia, horas in dias_fallo_lluvia.items():
    print(f"- {dia}: Faltan {horas} horas de datos")
    
# (Opcional) Si un día faltan exactamente 24 horas, significa que el sensor estuvo apagado el día entero.

🌡️ --- FALLO DE TEMPERATURA ---
Falta el dato del: 2025-01-01 00:00:00

🌧️ --- FALLOS DEL SENSOR DE LLUVIA ---
El sensor de lluvia falló en 13 días distintos. Resumen:
- 2024-08-28: Faltan 23 horas de datos
- 2024-08-29: Faltan 24 horas de datos
- 2024-08-30: Faltan 24 horas de datos
- 2024-08-31: Faltan 24 horas de datos
- 2024-09-01: Faltan 24 horas de datos
- 2024-09-02: Faltan 24 horas de datos
- 2024-09-03: Faltan 24 horas de datos
- 2024-09-04: Faltan 4 horas de datos
- 2024-12-10: Faltan 17 horas de datos
- 2024-12-11: Faltan 24 horas de datos
- 2024-12-12: Faltan 24 horas de datos
- 2024-12-13: Faltan 15 horas de datos
- 2025-01-01: Faltan 1 horas de datos


In [24]:
# --- 1. REPARACIÓN INTELIGENTE DE NULOS ---
# Rellenamos la lluvia vacía con 0.0
df_final['Lluvia'] = df_final['Lluvia'].fillna(0.0)

# Interpolamos la temperatura (traza una línea entre el valor anterior y el siguiente)
df_final['Temperatura'] = df_final['Temperatura'].interpolate()

# Por si ha quedado algún nulo en otra columna por error, hacemos una limpieza final
df_final = df_final.dropna()

In [ ]:
df_final.to_csv('../data/dataset_smart_campus_master.csv', index=False)

: 